# 딥러닝응용I(추천시스템)

**동덕여자대학교 데이터사이언스전공 유원상 교수**

**2026년 2학기**

## W01B · Colab 환경 구축과 첫 추천 앱

### 수업 개요
GitHub의 학생용 notebook을 Google Colab에서 실행하고, 공식 다운로드·검증된 수동 업로드·
synthetic sample 중 한 방법으로 세 데이터 파일을 준비합니다. 영화별 평균 평점 baseline을 만든 뒤
장르·최소 평점 수·Top-N을 선택할 수 있는 Gradio 웹 앱으로 연결합니다.

### 학습목표
1. GitHub notebook, Drive 사본, Colab runtime의 관계를 설명한다.
2. `u.user`, `u.item`, `u.data`를 올바른 구분자와 인코딩으로 읽는다.
3. 데이터 크기, 평점 범위와 ID 연결을 검사한다.
4. 평균 평점과 최소 평점 수로 비개인화 Top-N을 만든다.
5. Python callback을 Gradio component와 연결해 첫 추천 앱을 실행한다.


## 0. 실행 전에 꼭 읽기

1. GitHub 원본을 열었다면 먼저 **드라이브로 복사**합니다.
2. 코드 셀은 위에서 아래로 `Shift+Enter`로 실행합니다.
3. `NameError`가 나오면 필요한 변수를 만든 앞 셀이 실행되었는지 확인합니다.
4. Colab runtime을 다시 시작하면 변수와 `/content`의 파일이 사라질 수 있습니다.
5. MovieLens 원본 파일이나 사용자별 인구통계 표를 과제 파일에 넣지 않습니다.
6. `DATA_MODE="upload"`이면 `모두 실행` 도중 파일 선택 창에서 `ml-100k.zip`을 올립니다.


## 1. Python과 현재 폴더 확인

`Path.cwd()`는 코드가 현재 어느 폴더에서 실행되는지 알려 줍니다.
Colab의 기본 폴더는 보통 `/content`입니다. 자신의 Windows `C:` 드라이브가 아닙니다.


In [ ]:
from __future__ import annotations

import hashlib
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request
import zipfile
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import pandas as pd

IN_COLAB = "google.colab" in sys.modules
print("Python:", platform.python_version())
print("현재 폴더:", Path.cwd())
print("Colab runtime인가요?:", IN_COLAB)


## 2. GitHub 저장소를 runtime에 clone하기

`git clone`은 원격 repository의 사본을 현재 실행 환경으로 내려받습니다.
아래 셀은 Colab일 때만 `/content/recommender`에 공개 저장소를 clone합니다.
`--depth 1`은 오늘 확인할 최신 상태만 받아 다운로드를 줄입니다.

셀을 다시 실행하면 폴더가 이미 있는지 먼저 확인하므로 같은 저장소를 중복으로 clone하지 않습니다.


In [ ]:
REPO_URL = "https://github.com/lunalab-ai/recommender.git"
repo_dir = Path("/content/recommender")

if IN_COLAB:
    if repo_dir.exists():
        print("이미 clone되어 있습니다:", repo_dir)
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)],
            check=True,
        )
    print("저장소 파일 예:", sorted(path.name for path in repo_dir.iterdir()))
else:
    print("로컬 검증에서는 현재 authoring checkout을 사용하므로 clone을 건너뜁니다.")


## 3. Gradio 버전 준비

Colab에는 많은 package가 미리 설치되어 있지만 버전은 바뀔 수 있습니다.
다음 함수는 Gradio의 major version이 5 또는 6인지 확인하고, 필요할 때만 수업에서 검증한 범위를 설치합니다.

- `version("gradio")`: 설치된 버전 문자열을 읽습니다.
- `split(".")[0]`: `6.1.0`에서 첫 번째 숫자 `6`을 고릅니다.
- `sys.executable -m pip`: 지금 notebook을 실행하는 Python에 설치합니다.


In [ ]:
def ensure_gradio() -> str:
    """Ensure that the lesson uses a compatible Gradio major version."""
    try:
        installed = version("gradio")
    except PackageNotFoundError:
        installed = "0"
    major = int(installed.split(".")[0])
    if major not in {5, 6}:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "gradio>=5,<7"],
            check=True,
        )
        installed = version("gradio")
    return installed


GRADIO_VERSION = ensure_gradio()
print("Gradio:", GRADIO_VERSION)


## 4. MovieLens 100K 준비 방법 선택하기

오래된 MovieLens 100K의 사용 조건에는 별도 허가 없는 재배포 제한이 있습니다.
그래서 원본을 수업 GitHub에 넣지 않고 GroupLens 공식 URL에서 runtime으로 직접 받습니다.

먼저 `DATA_MODE`에서 오늘 사용할 방법 하나를 고릅니다.

- `"auto"`(기본값): 공식 URL 다운로드를 시도하고, 실패하면 synthetic sample로 전환합니다.
- `"upload"`: 본인이 공식 페이지에서 받은 `ml-100k.zip` 한 개를 Colab에 직접 올립니다.
- `"synthetic"`: 네트워크 없이 작은 가상 데이터로 바로 실습합니다.

`upload`를 고르면 이 셀을 실행할 때 파일 선택 창이 열립니다. 공식 ZIP이 아닌 파일이나 checksum이
다른 파일은 사용하지 않습니다. `auto`가 실패해도 TLS 검증을 끄는 코드는 사용하지 않습니다.


In [ ]:
DATA_MODE = "auto"  # Colab에서는 "auto", "upload", "synthetic" 중 하나로 바꾸세요.
DATA_MODE = os.getenv("RECOMMENDER_DATA_MODE", DATA_MODE).strip().lower()
VALID_DATA_MODES = {"auto", "upload", "synthetic"}
if DATA_MODE not in VALID_DATA_MODES:
    raise ValueError(f"DATA_MODE는 {sorted(VALID_DATA_MODES)} 중 하나여야 합니다: {DATA_MODE}")
print("선택한 데이터 준비 모드:", DATA_MODE)


## 5. Archive를 검증하고 필요한 세 파일만 꺼내기

자동 다운로드와 수동 업로드는 입구만 다르고, 이후 검사는 같습니다.

1. ZIP 전체 bytes의 MD5 checksum을 공식값과 비교합니다.
2. `ml-100k/u.user`, `u.item`, `u.data`가 각각 한 번씩 있는지 확인합니다.
3. 허용한 세 파일만 `/content/data/ml-100k`에 추출합니다.
4. 업로드하거나 임시로 받은 ZIP은 runtime에서 지웁니다.

자동 검증이나 로컬 환경에서는 `RECOMMENDER_DATA_DIR`와 `RECOMMENDER_UPLOAD_ARCHIVE`
환경변수로 준비된 폴더와 업로드 시험용 ZIP을 지정할 수 있습니다.


In [ ]:
DATA_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
EXPECTED_MD5 = "0e33842e24a9c977be4e0107933c0723"
NEEDED_FILES = ("u.user", "u.item", "u.data")


def has_lesson_files(path: Path) -> bool:
    """Return True when all three lesson files exist."""
    return all((path / name).is_file() for name in NEEDED_FILES)


def extract_verified_archive(archive_path: Path, root: Path) -> Path:
    """Check one archive and extract only the three files used today."""
    actual_md5 = hashlib.md5(
        archive_path.read_bytes(), usedforsecurity=False
    ).hexdigest()
    if actual_md5 != EXPECTED_MD5:
        raise ValueError(f"checksum 불일치: {actual_md5}")

    expected_members = {f"ml-100k/{name}" for name in NEEDED_FILES}
    data_dir = root / "ml-100k"
    with zipfile.ZipFile(archive_path) as archive:
        names = archive.namelist()
        missing = expected_members.difference(names)
        if missing:
            raise ValueError(f"archive에 필요한 파일이 없습니다: {sorted(missing)}")
        duplicates = sorted(member for member in expected_members if names.count(member) != 1)
        if duplicates:
            raise ValueError(f"archive에 중복 파일이 있습니다: {duplicates}")

        data_dir.mkdir(parents=True, exist_ok=True)
        for member in sorted(expected_members):
            target = data_dir / Path(member).name
            with archive.open(member) as source, target.open("wb") as output:
                shutil.copyfileobj(source, output)
    return data_dir


def download_movielens_data(root: Path) -> Path:
    """Reuse complete local files or download the official archive."""
    if has_lesson_files(root):
        return root
    if has_lesson_files(root / "ml-100k"):
        return root / "ml-100k"

    root.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=root, suffix=".zip", delete=False) as temp_file:
        archive_path = Path(temp_file.name)
    try:
        with urllib.request.urlopen(DATA_URL, timeout=60) as response:
            with archive_path.open("wb") as output:
                shutil.copyfileobj(response, output)
        return extract_verified_archive(archive_path, root)
    finally:
        archive_path.unlink(missing_ok=True)


def upload_movielens_data(root: Path) -> Path:
    """Upload one official archive in Colab, then verify and extract it."""
    if not IN_COLAB:
        configured_archive = os.getenv("RECOMMENDER_UPLOAD_ARCHIVE")
        if not configured_archive:
            raise RuntimeError("로컬 upload 검증에는 RECOMMENDER_UPLOAD_ARCHIVE가 필요합니다.")
        return extract_verified_archive(Path(configured_archive), root)

    from google.colab import files

    print("파일 선택 창에서 ml-100k.zip 한 개만 선택하세요.")
    uploaded = files.upload()
    if set(uploaded) != {"ml-100k.zip"}:
        raise ValueError("파일명을 ml-100k.zip으로 확인하고 한 개만 다시 업로드하세요.")
    uploaded_path = Path("ml-100k.zip")
    uploaded_path.write_bytes(uploaded["ml-100k.zip"])
    try:
        return extract_verified_archive(uploaded_path, root)
    finally:
        uploaded_path.unlink(missing_ok=True)


def write_generated_synthetic_data(path: Path) -> Path:
    """Create a deterministic CC0 dataset when the bundled sample is unavailable."""
    path.mkdir(parents=True, exist_ok=True)
    occupations = [
        "student", "student", "engineer", "artist", "scientist",
        "writer", "educator", "programmer", "designer", "researcher",
    ]
    user_lines = [
        f"{user_id}|{19 + user_id}|{'F' if user_id % 2 else 'M'}|{occupation}|00000"
        for user_id, occupation in enumerate(occupations, start=1)
    ]

    genre_names = [
        "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy",
        "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
        "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western",
    ]
    movie_specs = [
        ("Starlight Library", "Drama"), ("Robot's Day", "Sci-Fi"),
        ("Summer Recipe", "Romance"), ("Data Detective", "Mystery"),
        ("Concert Across Sea", "Musical"), ("Friends on Tiny Planet", "Animation"),
        ("Last Algorithm", "Thriller"), ("Alley Camera", "Documentary"),
        ("Train Above Clouds", "Fantasy"), ("Midnight Comedy", "Comedy"),
        ("Green City", "Documentary"), ("Map of Memory", "Drama"),
    ]
    item_lines = []
    for movie_id, (title, genre) in enumerate(movie_specs, start=1):
        flags = ["1" if name == genre else "0" for name in genre_names]
        fields = [
            str(movie_id), f"{title} (2026)", f"{movie_id:02d}-Jan-2026", "", "", *flags,
        ]
        item_lines.append("|".join(fields))

    rating_lines = []
    timestamp = 1
    for user_id in range(1, 11):
        for offset in range(6):
            movie_id = ((user_id - 1 + 2 * offset) % 12) + 1
            rating = 3 + ((user_id + movie_id + offset) % 3)
            rating_lines.append(f"{user_id}\t{movie_id}\t{rating}\t{timestamp}")
            timestamp += 1

    (path / "u.user").write_text("\n".join(user_lines) + "\n", encoding="latin-1")
    (path / "u.item").write_text("\n".join(item_lines) + "\n", encoding="latin-1")
    (path / "u.data").write_text("\n".join(rating_lines) + "\n", encoding="latin-1")
    return path


def synthetic_data_dir(root: Path) -> tuple[Path, str]:
    """Use the bundled synthetic sample or generate an equivalent local one."""
    default_path = (
        repo_dir / "data" / "sample" / "ml100k_tiny"
        if IN_COLAB
        else Path.cwd().parents[1] / "data" / "sample" / "ml100k_tiny"
    )
    bundled_path = Path(os.getenv("RECOMMENDER_SYNTHETIC_DIR", default_path))
    if has_lesson_files(bundled_path):
        return bundled_path, "bundled-synthetic"

    generated_path = write_generated_synthetic_data(root / "ml100k_tiny_generated")
    print("저장소 sample이 없어 notebook 내장 규칙으로 synthetic 데이터를 만들었습니다.")
    return generated_path, "generated-synthetic"


configured_root = Path(os.getenv("RECOMMENDER_DATA_DIR", "/content/data"))
download_started = time.perf_counter()
if DATA_MODE == "upload":
    data_dir = upload_movielens_data(configured_root)
    DATASET_KIND = "official"
    DATA_SOURCE = "manual-upload"
elif DATA_MODE == "synthetic":
    data_dir, synthetic_source = synthetic_data_dir(configured_root)
    DATASET_KIND = "synthetic-fallback"
    DATA_SOURCE = f"selected-{synthetic_source}"
else:
    try:
        data_dir = download_movielens_data(configured_root)
        DATASET_KIND = "official"
        DATA_SOURCE = "official-download-or-cache"
    except (urllib.error.URLError, TimeoutError, ValueError, zipfile.BadZipFile) as error:
        data_dir, synthetic_source = synthetic_data_dir(configured_root)
        DATASET_KIND = "synthetic-fallback"
        DATA_SOURCE = f"automatic-{synthetic_source}-fallback"
        print("공식 서버에 안전하게 연결하지 못해 synthetic fallback을 사용합니다.")
        print("원인:", error)
download_seconds = time.perf_counter() - download_started

print("데이터 종류:", DATASET_KIND)
print("데이터 출처:", DATA_SOURCE)
print("데이터 폴더:", data_dir)
print("준비 시간(초):", round(download_seconds, 3))
print("파일:", [path.name for path in sorted(data_dir.iterdir()) if path.is_file()])


### 준비가 실패했을 때

오류 메시지의 마지막 줄을 먼저 읽습니다.

- `URLError`, `TimeoutError`: 수업용 synthetic fallback으로 전환되었는지 확인합니다.
- `checksum 불일치`: 자동 다운로드가 불완전했거나 업로드 파일이 공식 ZIP과 다릅니다.
- `BadZipFile`: runtime을 다시 시작하고 다운로드 셀부터 다시 실행합니다.
- 저장소 sample이 없으면 notebook이 `/content/data/ml100k_tiny_generated`를 자동 생성합니다.
- 업로드 창이 기다리는 중이면 `ml-100k.zip` 한 개를 선택하고 업로드가 끝날 때까지 기다립니다.

2026-09-03 로컬 검증에서는 공식 file server의 TLS 인증서 만료를 확인했습니다. Notebook은 TLS
검증을 끄거나 제3자 mirror를 사용하지 않습니다. 전체 데이터를 이미 받았다면 `upload`, 빠르게
계속하려면 `synthetic`을 선택합니다. 화면의 `데이터 종류`와 `데이터 출처`를 함께 확인하세요.


## 6. 세 파일의 열 이름 준비

MovieLens 100K의 세 파일에는 첫 줄 header가 없습니다. 따라서 우리가 열 이름을 순서대로 지정합니다.
`list`는 순서가 있는 값의 묶음이고, `*GENRES`는 장르 목록의 각 값을 다른 목록 안에 펼칩니다.


In [ ]:
USER_COLUMNS = ["user_id", "age", "sex", "occupation", "zip_code"]
GENRES = [
    "unknown",
    "Action",
    "Adventure",
    "Animation",
    "Children's",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "Film-Noir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "Sci-Fi",
    "Thriller",
    "War",
    "Western",
]
ITEM_COLUMNS = [
    "movie_id",
    "title",
    "release_date",
    "video_release_date",
    "imdb_url",
    *GENRES,
]
RATING_COLUMNS = ["user_id", "movie_id", "rating", "timestamp"]

print("u.user 열 수:", len(USER_COLUMNS))
print("u.item 열 수:", len(ITEM_COLUMNS))
print("u.data 열 수:", len(RATING_COLUMNS))


## 7. `u.user` 읽기

`u.user`는 `|`로 값이 나뉘고 `latin-1`로 인코딩되어 있습니다.

- `pd.read_csv`: 구분된 표 파일을 DataFrame으로 읽습니다.
- `sep="|"`: 한 줄에서 열을 나누는 문자를 지정합니다.
- `names=USER_COLUMNS`: header가 없는 파일에 열 이름을 붙입니다.
- `encoding="latin-1"`: bytes를 문자로 해석하는 방법을 지정합니다.

우편번호는 row-level로 출력하지 않고 구조 확인에만 사용합니다.


In [ ]:
users = pd.read_csv(
    data_dir / "u.user",
    sep="|",
    names=USER_COLUMNS,
    encoding="latin-1",
)
print("users shape:", users.shape)
display(users[["user_id", "age", "sex", "occupation"]].head(3))


## 8. `u.item` 읽기

`u.item`도 `|`로 나뉩니다. 앞의 다섯 열은 ID, 제목, 날짜, URL이고 뒤의 열은 장르입니다.
장르 값 `1`은 해당 장르에 속한다는 뜻이고 `0`은 속하지 않는다는 뜻입니다.
한 영화가 둘 이상의 장르에서 `1`을 가질 수 있습니다.


In [ ]:
movies = pd.read_csv(
    data_dir / "u.item",
    sep="|",
    names=ITEM_COLUMNS,
    encoding="latin-1",
)
print("movies shape:", movies.shape)
display(movies[["movie_id", "title", "release_date", "Action", "Comedy"]].head(3))


## 9. `u.data` 읽기

`u.data`는 앞의 두 파일과 달리 tab 문자로 나뉩니다. Python 문자열에서 tab은 `"\t"`로 씁니다.
한 행은 한 사용자가 한 영화에 남긴 1–5점 평점과 평가 시점을 뜻합니다.


In [ ]:
ratings = pd.read_csv(
    data_dir / "u.data",
    sep="\t",
    names=RATING_COLUMNS,
    encoding="latin-1",
)
print("ratings shape:", ratings.shape)
display(ratings.head(3))


## 10. 실행 성공과 데이터 정상 여부를 구분하기

오류 없이 표가 만들어져도 separator나 열 순서가 틀릴 수 있습니다.
`assert`는 반드시 만족해야 하는 조건을 검사하며, 거짓이면 즉시 실행을 멈춥니다.

- `.between(1, 5)`: 각 평점이 범위 안인지 확인합니다.
- `.isin(...)`: rating의 ID가 사용자·영화 표에 실제로 있는지 확인합니다.
- `.all()`: 모든 행이 조건을 만족하는지 확인합니다.

공식 데이터는 `(943, 1682, 100000)`, synthetic fallback은 `(10, 12, 60)`이어야 합니다.


In [ ]:
expected_sizes = (943, 1682, 100000) if DATASET_KIND == "official" else (10, 12, 60)
assert (len(users), len(movies), len(ratings)) == expected_sizes
assert users["user_id"].is_unique
assert movies["movie_id"].is_unique
assert ratings["rating"].between(1, 5).all()
assert ratings["user_id"].isin(users["user_id"]).all()
assert ratings["movie_id"].isin(movies["movie_id"]).all()

print("데이터 크기, ID 중복, 평점 범위와 ID 연결 검사를 통과했습니다.")


### 활동 1 · separator 디버깅

아래 코드는 `u.data`에 일부러 잘못된 separator를 사용합니다. 실행 결과의 shape와 첫 행을 보고
`separator_for_u_data` 한 줄을 올바르게 고치세요. 오류가 없다는 사실만으로 데이터가 정상이라고
판단하면 안 됩니다.


In [ ]:
separator_for_u_data = "|"  # TODO: u.data의 실제 구분자로 고치세요.
broken_ratings = pd.read_csv(
    data_dir / "u.data",
    sep=separator_for_u_data,
    names=RATING_COLUMNS,
    encoding="latin-1",
)
print("현재 separator로 읽은 shape:", broken_ratings.shape)
print("rating 열에서 읽힌 값 수:", broken_ratings["rating"].notna().sum())
display(broken_ratings.head(3))


## 11. 영화별 평점 수와 평균 계산

`ratings`에는 제목이 없고 `movies`에는 개별 평점이 없습니다. 다음 method chain은 평점을 영화별로
묶어 개수와 평균을 계산한 뒤, 같은 `movie_id`를 기준으로 제목과 장르를 붙입니다.

1. `groupby`: 같은 영화 ID의 행을 묶습니다.
2. `agg`: 각 묶음의 개수와 평균을 계산합니다.
3. `merge`: 두 표의 같은 영화 ID를 연결합니다.


In [ ]:
aggregation_started = time.perf_counter()
movie_stats = (
    ratings.groupby("movie_id", as_index=False)["rating"]
    .agg(rating_count="count", mean_rating="mean")
    .merge(movies[["movie_id", "title", *GENRES]], on="movie_id", how="inner")
)
aggregation_seconds = time.perf_counter() - aggregation_started
DEFAULT_MIN_RATINGS = 50 if DATASET_KIND == "official" else 3
MAX_MIN_RATINGS = 200 if DATASET_KIND == "official" else 5

print("movie_stats shape:", movie_stats.shape)
print("집계 시간(초):", round(aggregation_seconds, 4))
display(movie_stats[["movie_id", "title", "rating_count", "mean_rating"]].head())


## 12. 평균 평점 baseline 추천 함수

평균 평점만 정렬하면 한 명에게 5점을 받은 영화가 상위에 나타날 수 있습니다. 오늘은 일정 수 이상의
평점을 받은 영화만 후보로 남기는 `min_ratings` 조건을 함께 관찰합니다.

정렬 기준은 평균 평점 내림차순, 평점 수 내림차순, 제목 오름차순, 영화 ID 오름차순입니다.
마지막 두 기준은 같은 입력에서 같은 결과가 나오도록 tie를 결정합니다.


In [ ]:
def recommend_movies(
    genre: str = "전체",
    min_ratings: int = 50,
    top_n: int = 10,
) -> pd.DataFrame:
    """Return a deterministic non-personalized MovieLens baseline."""
    if genre != "전체" and genre not in GENRES:
        raise ValueError(f"알 수 없는 장르입니다: {genre}")
    if min_ratings < 1:
        raise ValueError("min_ratings는 1 이상이어야 합니다.")
    if top_n < 1:
        raise ValueError("top_n은 1 이상이어야 합니다.")

    candidates = movie_stats.copy()
    if genre != "전체":
        candidates = candidates.loc[candidates[genre].eq(1)]
    candidates = candidates.loc[candidates["rating_count"] >= min_ratings]

    ranked = candidates.sort_values(
        ["mean_rating", "rating_count", "title", "movie_id"],
        ascending=[False, False, True, True],
        kind="mergesort",
    ).head(top_n)
    result = ranked[["movie_id", "title", "mean_rating", "rating_count"]].copy()
    result["mean_rating"] = result["mean_rating"].round(3)
    return result.reset_index(drop=True)


display(recommend_movies(genre="전체", min_ratings=DEFAULT_MIN_RATINGS, top_n=10))


### 결과 예측

실행 전에 다음을 예상해 보세요.

1. `min_ratings`를 1로 낮추면 평균 5.0 영화가 더 많이 나타날까요?
2. `min_ratings`를 100으로 높이면 후보 수는 늘어날까요, 줄어들까요?
3. `genre="Comedy"`이면 어느 장르 열의 값이 1인 영화만 남을까요?


In [ ]:
prediction = {
    "min_ratings_1": "",  # TODO: 예상 변화를 한 문장으로 적으세요.
    "min_ratings_100": "",  # TODO: 예상 변화를 한 문장으로 적으세요.
}
prediction


### 활동 2 · 최소 평점 수 비교

아래 목록에 1, 20, 50, 100을 차례로 넣어 결과의 첫 영화와 후보 수 변화를 관찰합니다.
`for`는 목록의 값을 하나씩 꺼내 같은 코드를 반복합니다.


In [ ]:
comparison_thresholds = [1, 20, 50, 100] if DATASET_KIND == "official" else [1, 2, 3, 5]
for minimum in comparison_thresholds:
    result = recommend_movies(min_ratings=minimum, top_n=5)
    print(f"min_ratings={minimum}: 반환된 영화 {len(result)}개")
    display(result)


## 13. GUI가 호출할 callback

Callback은 화면의 값을 Python 추천 함수로 전달하고, 결과를 화면에 맞는 형태로 돌려줍니다.
Slider 값은 환경에 따라 숫자 표현이 달라질 수 있어 `int(...)`로 변환합니다. 내부 계산의 영문 열
이름은 유지하고 화면으로 보내기 직전에 한글로 바꿉니다.


In [ ]:
def recommend_for_app(genre: str, min_ratings: int, top_n: int) -> pd.DataFrame:
    """Adapt UI inputs and return a display-friendly table."""
    result = recommend_movies(
        genre=genre,
        min_ratings=int(min_ratings),
        top_n=int(top_n),
    )
    return result.rename(
        columns={
            "movie_id": "영화 ID",
            "title": "영화 제목",
            "mean_rating": "평균 평점",
            "rating_count": "평점 수",
        }
    )


callback_started = time.perf_counter()
callback_test = recommend_for_app("Comedy", DEFAULT_MIN_RATINGS, 5)
callback_seconds = time.perf_counter() - callback_started
assert callback_test.columns.tolist() == ["영화 ID", "영화 제목", "평균 평점", "평점 수"]
assert len(callback_test) <= 5

print("callback 응답 시간(초):", round(callback_seconds, 6))
display(callback_test)


## 14. Gradio GUI 만들기

- `Blocks`: 앱 전체 영역
- `Row`: 입력 component를 한 행에 배치
- `Dropdown`, `Slider`: callback에 보낼 입력
- `Button`: 계산을 요청하는 event
- `Dataframe`: callback 결과를 표시할 출력
- `.click`: 버튼, callback, 입력과 출력을 연결

아래 셀은 앱의 구조만 만들고 아직 server를 실행하지 않습니다. 따라서 GUI 문제를 추천 계산 문제와
분리해 확인할 수 있습니다.


In [ ]:
import gradio as gr

with gr.Blocks(title="LUNA 영화 추천 실험실", analytics_enabled=False) as demo:
    gr.Markdown(
        "# 🎬 LUNA 영화 추천 실험실\n"
        "장르와 조건을 선택해 평균 평점 기반 추천을 관찰하세요."
    )
    with gr.Row():
        genre_input = gr.Dropdown(
            choices=["전체", *GENRES[1:]],
            value="전체",
            label="장르",
        )
        min_ratings_input = gr.Slider(
            minimum=1,
            maximum=MAX_MIN_RATINGS,
            value=DEFAULT_MIN_RATINGS,
            step=1,
            label="최소 평점 수",
        )
        top_n_input = gr.Slider(
            minimum=1,
            maximum=20,
            value=10,
            step=1,
            label="추천 개수",
        )
    run_button = gr.Button("추천 영화 보기", variant="primary")
    result_output = gr.Dataframe(
        headers=["영화 ID", "영화 제목", "평균 평점", "평점 수"],
        datatype=["number", "str", "number", "number"],
        interactive=False,
        label="추천 결과",
    )
    gr.Markdown(
        "이 결과는 모든 사용자에게 같은 집계 기준을 적용하는 첫 baseline입니다. "
        "개인 취향을 반영하지 않습니다."
    )
    run_button.click(
        fn=recommend_for_app,
        inputs=[genre_input, min_ratings_input, top_n_input],
        outputs=result_output,
    )

print("Gradio Blocks가 만들어졌습니다:", type(demo).__name__)


## 15. 앱 실행

Colab에서는 다음 셀이 앱과 임시 share link를 만듭니다. 링크는 실행 중인 runtime으로 연결되는
tunnel이며 영구 서비스가 아닙니다. runtime이 종료되면 링크도 작동하지 않습니다.
링크를 아는 사람이 접근할 수 있으므로 개인 정보나 비공개 데이터를 입력하지 않습니다.

자동화된 로컬 notebook 검증에서는 server를 띄우지 않고 구조와 callback만 검사합니다.


In [ ]:
if IN_COLAB:
    launch_result = demo.launch(share=True, debug=False)
else:
    print("로컬 자동 검증에서는 launch를 생략했습니다. 별도 smoke test에서 server를 확인합니다.")


### 활동 3 · GUI 한 곳 바꾸기

다음 중 하나를 선택해 위 GUI 셀을 수정한 뒤 다시 실행하세요.

- 버튼 문구를 `나의 첫 추천 실행`으로 변경
- 최소 평점 수 기본값을 100으로 변경
- 추천 개수 기본값을 5로 변경

Colab에서 이미 앱을 실행했다면 먼저 아래 셀로 닫고 GUI 셀부터 다시 실행합니다.


In [ ]:
if IN_COLAB:
    # GUI를 수정하기 전 주석을 지우고 실행하세요.
    # demo.close()
    pass


## 16. 앱 오류를 네 계층으로 나누기

1. Data: `users`, `movies`, `ratings`가 올바른 shape인가?
2. Ranking: `recommend_movies(...)`가 표를 반환하는가?
3. Callback: `recommend_for_app(...)`가 한글 열 이름의 표를 반환하는가?
4. GUI: component와 `.click(...)`의 입출력 순서가 올바른가?

화면이 열리지 않더라도 1–3이 통과하면 추천 계산은 정상입니다. 마지막 오류만 보고 전체 코드를
다시 쓰지 말고, 어느 계층까지 정상인지 확인합니다.


## 점검 퀴즈

1. GitHub repository와 Colab runtime은 각각 무엇을 보관하나요?
2. runtime을 다시 시작한 뒤 `ratings` 변수가 사라지는 이유는 무엇인가요?
3. MovieLens 100K 원본을 수업 GitHub에 넣지 않는 이유는 무엇인가요?
4. `u.user`, `u.item`, `u.data`는 각각 무엇을 나타내나요?
5. `u.data`의 `sep="\t"`에서 `\t`는 무엇인가요?
6. 평균 평점만 정렬할 때 평점 수가 매우 적은 영화가 문제가 될 수 있는 이유는 무엇인가요?
7. `run_button.click`에서 callback, inputs, outputs는 어떻게 연결되나요?
8. 오늘 앱이 개인화 추천이 아닌 이유는 무엇인가요?


## Take-home message

- Notebook 문서와 runtime의 실행 상태는 다르므로 위에서 아래로 실행하고 Drive 사본을 저장합니다.
- 공개 GitHub에는 코드만 두고 MovieLens 원본은 공식 서버에서 runtime으로 직접 받습니다.
- 파일을 읽은 직후 shape, 열 이름, 값 범위와 ID 연결을 확인합니다.
- 평균 평점 baseline은 간단한 출발점이며 최소 평점 수에 따라 결과가 달라집니다.
- Data, ranking, callback, GUI를 분리하면 같은 앱을 이후 알고리즘으로 발전시킬 수 있습니다.

## 참고자료

- GroupLens, MovieLens 100K: https://grouplens.org/datasets/movielens/100k/
- MovieLens 100K README: https://files.grouplens.org/datasets/movielens/ml-100k-README.txt
- Google Colab FAQ: https://research.google.com/colaboratory/intl/en-GB/faq.html
- GitHub, Cloning a repository: https://docs.github.com/en/repositories/creating-and-managing-repositories/cloning-a-repository
- Python 한국어 자습서: https://docs.python.org/ko/3/tutorial/
- pandas Getting started: https://pandas.pydata.org/docs/getting_started/intro_tutorials/index.html
- Gradio Blocks: https://www.gradio.app/docs/gradio/blocks
- Gradio share links: https://www.gradio.app/guides/understanding-gradio-share-links
- 임일, 『AI 에이전트를 위한 개인화 추천 알고리즘: Python, 머신러닝, AI, LLM 활용』,
  도서출판청람, 2025, 2장, pp. 12–17.

교재에서 가져온 범위는 MovieLens 세 파일의 구조와 평균 평점 기반 추천의 출발점입니다.
최소 평점 수 비교, 오류 진단, GUI, callback과 활동은 수업을 위해 독자적으로 구성했습니다.
